# Hybrid CNN-RNN models for assigned windows

This notebook trains hybrid neural networks that combine **Conv1D** layers with **LSTM/GRU** layers.

Assigned windows:

| Input window | Output window |
|---:|---:|
| 10 | 30 |
| 10 | 90 |
| 30 | 1 |
| 30 | 5 |

The goal is to evaluate mixed architectures on the forecasting task and export results that can be used directly in the final report.

The test set is not used for model selection. For each run, the notebook uses:

1. train split for fitting the model,
2. validation split for early stopping and model comparison,
3. test split for final evaluation.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import sys
import json
from pathlib import Path
from datetime import datetime

# Locate project root from notebook execution directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
PROJECT_ROOT = next(p for p in _candidates if (p / "util.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

import keras
from keras import backend as K
from keras.models import Model
from keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    SpatialDropout1D,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout,
)
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from util import get_train_test, RANDOM_SEED, plot_training_curve

try:
    from util import configure_mlflow
except ImportError:
    import mlflow

    def configure_mlflow(experiment_name: str):
        tracking_uri = f"sqlite:///{PROJECT_ROOT / 'model' / 'mlflow.db'}"
        mlflow.set_tracking_uri(tracking_uri)
        mlflow.set_experiment(experiment_name)
        return mlflow


np.random.seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

DATA_OUT = PROJECT_ROOT / "data" / "mixtos" / "cnn_rnn_hybrid"
HISTORY_DIR = DATA_OUT / "history"
PLOTS_DIR = DATA_OUT / "plots"

DATA_OUT.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

mlflow = configure_mlflow("hybrid_cnn_rnn_models")

FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "0") == "1"
LOG_MODEL_ARTIFACT = os.getenv("LOG_MODEL_ARTIFACT", "0") == "1"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FAST_DEV_RUN:", FAST_DEV_RUN)
print("LOG_MODEL_ARTIFACT:", LOG_MODEL_ARTIFACT)

## Configuration

The full run trains three hybrid architectures for each assigned window:

- `CNN_LSTM`
- `CNN_GRU`
- `CNN_BiGRU`

The hyperparameters are intentionally controlled and comparable across windows. This avoids an excessively large search while still producing a clean comparison between hybrid architectures.

A fast smoke test can be launched from terminal with:

```bash
FAST_DEV_RUN=1 jupyter nbconvert --to notebook --execute model/mixtos/cnn_rnn_hybrid/01_hybrid_cnn_rnn_grid.ipynb \
  --output 01_hybrid_cnn_rnn_grid_executed.ipynb \
  --output-dir model/mixtos/cnn_rnn_hybrid/outputs
```

In [ ]:
WINDOWS = [
    (10, 30),
    (10, 90),
    (30, 1),
    (30, 5),
]

ARCHITECTURES = [
    {
        "architecture": "CNN_LSTM",
        "filters": 64,
        "kernel_size": 3,
        "rnn_units": 64,
        "dense_units": 64,
        "spatial_dropout": 0.10,
        "dropout": 0.15,
        "learning_rate": 3e-4,
        "batch_size": 128,
    },
    {
        "architecture": "CNN_GRU",
        "filters": 64,
        "kernel_size": 3,
        "rnn_units": 64,
        "dense_units": 64,
        "spatial_dropout": 0.10,
        "dropout": 0.15,
        "learning_rate": 3e-4,
        "batch_size": 128,
    },
    {
        "architecture": "CNN_BiGRU",
        "filters": 64,
        "kernel_size": 5,
        "rnn_units": 64,
        "dense_units": 64,
        "spatial_dropout": 0.10,
        "dropout": 0.20,
        "learning_rate": 3e-4,
        "batch_size": 128,
    },
]

MAX_EPOCHS = 80
EARLY_STOPPING_PATIENCE = 10
LR_PATIENCE = 5
VALIDATION_RATIO = 0.10

if FAST_DEV_RUN:
    WINDOWS = WINDOWS[:1]
    ARCHITECTURES = ARCHITECTURES[:1]
    MAX_EPOCHS = 2
    EARLY_STOPPING_PATIENCE = 1
    LR_PATIENCE = 1

print("Windows:", WINDOWS)
print("Architectures:", [cfg["architecture"] for cfg in ARCHITECTURES])
print("Max epochs:", MAX_EPOCHS)

## Data preparation

The function `get_train_test` returns data with the sequence format required by recurrent and convolutional models:

```text
X: samples × input_window × assets
y: samples × assets
```

The validation set is taken from the end of the training set. Inputs are standardized using only the training split to avoid data leakage.

In [ ]:
def split_train_val(X_train, y_train, val_ratio=VALIDATION_RATIO):
    val_size = int(len(X_train) * val_ratio)
    if val_size <= 0:
        raise ValueError("Validation split is empty. Increase the training size or validation ratio.")

    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_train_final = X_train[:-val_size]
    y_train_final = y_train[:-val_size]
    return X_train_final, y_train_final, X_val, y_val


def scale_X_only(X_train, X_val, X_test):
    """Scale only the inputs. The target remains in the original return scale."""
    n_train, window, n_assets = X_train.shape
    n_val = X_val.shape[0]
    n_test = X_test.shape[0]

    scaler = StandardScaler()

    X_train_2d = X_train.reshape(n_train, -1)
    X_val_2d = X_val.reshape(n_val, -1)
    X_test_2d = X_test.reshape(n_test, -1)

    X_train_scaled = scaler.fit_transform(X_train_2d).reshape(n_train, window, n_assets)
    X_val_scaled = scaler.transform(X_val_2d).reshape(n_val, window, n_assets)
    X_test_scaled = scaler.transform(X_test_2d).reshape(n_test, window, n_assets)

    return X_train_scaled, X_val_scaled, X_test_scaled


def load_window_data(input_window, output_window):
    d = get_train_test(
        input_window_size=input_window,
        output_window_size=output_window,
    )

    X_train_raw, y_train_raw = d.X_train, d.y_train
    X_test_raw, y_test = d.X_test, d.y_test

    X_train_raw, y_train, X_val_raw, y_val = split_train_val(X_train_raw, y_train_raw)
    X_train, X_val, X_test = scale_X_only(X_train_raw, X_val_raw, X_test_raw)

    return X_train, y_train, X_val, y_val, X_test, y_test

## Hybrid model builder

The hybrid models first use a convolutional layer to detect local temporal patterns. The recurrent block then models the sequential component of the window. The final dense layers map the learned representation to the 23 output assets.

In [ ]:
def build_hybrid_model(input_window, n_assets, cfg):
    architecture = cfg["architecture"]

    inputs = Input(shape=(input_window, n_assets))

    x = Conv1D(
        filters=cfg["filters"],
        kernel_size=cfg["kernel_size"],
        padding="causal",
        activation="relu",
    )(inputs)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(cfg["spatial_dropout"])(x)

    if architecture == "CNN_LSTM":
        x = LSTM(cfg["rnn_units"], return_sequences=False)(x)
    elif architecture == "CNN_GRU":
        x = GRU(cfg["rnn_units"], return_sequences=False)(x)
    elif architecture == "CNN_BiGRU":
        x = Bidirectional(GRU(cfg["rnn_units"], return_sequences=False))(x)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    x = Dense(cfg["dense_units"], activation="relu")(x)
    x = Dropout(cfg["dropout"])(x)

    outputs = Dense(n_assets, activation="linear")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["learning_rate"]),
        loss="mae",
        metrics=["mae"],
    )
    return model

## Training and MLflow logging

Each run is logged to MLflow with the window, architecture, hyperparameters, metrics and training curve. CSV files are also generated under `data/mixtos/cnn_rnn_hybrid/` for easier inclusion in the report.

In [ ]:
def safe_run_name(architecture, input_window, output_window):
    return f"hybrid_{architecture}_input{input_window}_output{output_window}"


def delete_existing_mlflow_run(run_name):
    existing_runs = mlflow.search_runs(filter_string=f'tags.mlflow.runName = "{run_name}"')
    if not existing_runs.empty:
        for run_id in existing_runs["run_id"]:
            mlflow.delete_run(run_id)


def save_history_and_plot(history, run_name):
    history_df = pd.DataFrame(history.history)
    history_path = HISTORY_DIR / f"{run_name}_history.csv"
    plot_path = PLOTS_DIR / f"{run_name}_loss_curve.png"

    history_df.to_csv(history_path, index=False)

    fig = plot_training_curve(history)
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return history_path, plot_path


def train_one_model(input_window, output_window, cfg):
    K.clear_session()
    keras.utils.set_random_seed(RANDOM_SEED)

    run_name = safe_run_name(cfg["architecture"], input_window, output_window)
    delete_existing_mlflow_run(run_name)

    X_train, y_train, X_val, y_val, X_test, y_test = load_window_data(input_window, output_window)
    n_assets = X_train.shape[2]

    model = build_hybrid_model(input_window, n_assets, cfg)

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            min_delta=1e-6,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=LR_PATIENCE,
            min_lr=1e-6,
        ),
    ]

    print()
    print("=" * 90)
    print(f"Training {run_name}")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
    print("X_test: ", X_test.shape, "y_test: ", y_test.shape)
    print("Params:", model.count_params())

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=cfg["batch_size"],
        callbacks=callbacks,
        verbose=1,
        shuffle=True,
    )

    y_pred_train = model.predict(X_train, verbose=0)
    y_pred_val = model.predict(X_val, verbose=0)
    y_pred_test = model.predict(X_test, verbose=0)

    row = {
        "model": cfg["architecture"],
        "input_window": input_window,
        "output_window": output_window,
        "MAE_train": mean_absolute_error(y_train, y_pred_train),
        "MAE_val": mean_absolute_error(y_val, y_pred_val),
        "MAE_test": mean_absolute_error(y_test, y_pred_test),
        "params": model.count_params(),
        "epochs_trained": len(history.history["loss"]),
        "filters": cfg["filters"],
        "kernel_size": cfg["kernel_size"],
        "rnn_units": cfg["rnn_units"],
        "dense_units": cfg["dense_units"],
        "spatial_dropout": cfg["spatial_dropout"],
        "dropout": cfg["dropout"],
        "learning_rate": cfg["learning_rate"],
        "batch_size": cfg["batch_size"],
    }

    history_path, plot_path = save_history_and_plot(history, run_name)
    row["history_path"] = str(history_path.relative_to(PROJECT_ROOT))
    row["plot_path"] = str(plot_path.relative_to(PROJECT_ROOT))

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_family", "Hybrid_CNN_RNN")
        mlflow.set_tag("model_name", cfg["architecture"])
        mlflow.log_params({
            "input_window_size": input_window,
            "output_window_size": output_window,
            "architecture": cfg["architecture"],
            "filters": cfg["filters"],
            "kernel_size": cfg["kernel_size"],
            "rnn_units": cfg["rnn_units"],
            "dense_units": cfg["dense_units"],
            "spatial_dropout": cfg["spatial_dropout"],
            "dropout": cfg["dropout"],
            "learning_rate": cfg["learning_rate"],
            "batch_size": cfg["batch_size"],
            "epochs_trained": row["epochs_trained"],
            "n_params": row["params"],
            "validation_ratio": VALIDATION_RATIO,
        })

        for epoch, (loss, val_loss) in enumerate(zip(history.history["loss"], history.history["val_loss"]), start=1):
            mlflow.log_metric("train_loss", float(loss), step=epoch)
            mlflow.log_metric("val_loss", float(val_loss), step=epoch)

        mlflow.log_metric("train_mae", float(row["MAE_train"]))
        mlflow.log_metric("val_mae", float(row["MAE_val"]))
        mlflow.log_metric("test_mae", float(row["MAE_test"]))
        mlflow.log_artifact(str(history_path), artifact_path="history")
        mlflow.log_artifact(str(plot_path), artifact_path="plots")

        if LOG_MODEL_ARTIFACT:
            mlflow.keras.log_model(model, name=f"{run_name}_model")

    print("Result:", json.dumps({k: v for k, v in row.items() if not k.endswith('_path')}, indent=2))
    return row


## Execute grid

The full run trains 12 models:

```text
4 windows × 3 architectures = 12 hybrid models
```

The best model for each window is selected by validation MAE.

In [ ]:
rows = []

for input_window, output_window in WINDOWS:
    for cfg in ARCHITECTURES:
        row = train_one_model(input_window, output_window, cfg)
        rows.append(row)

        partial = pd.DataFrame(rows)
        partial.to_csv(DATA_OUT / "hybrid_all_results_partial.csv", index=False)

results = pd.DataFrame(rows)
results = results.sort_values(["input_window", "output_window", "MAE_val"]).reset_index(drop=True)
results_path = DATA_OUT / "hybrid_all_results.csv"
results.to_csv(results_path, index=False)

best_by_window = (
    results.sort_values("MAE_val")
    .groupby(["input_window", "output_window"], as_index=False)
    .first()
    .sort_values(["input_window", "output_window"])
)
best_path = DATA_OUT / "hybrid_best_by_window.csv"
best_by_window.to_csv(best_path, index=False)

print("All results saved to:", results_path)
print("Best by window saved to:", best_path)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.8f}".format)

display(results[[
    "model", "input_window", "output_window", "MAE_train", "MAE_val", "MAE_test", "params", "epochs_trained"
]])

display(best_by_window[[
    "model", "input_window", "output_window", "MAE_train", "MAE_val", "MAE_test", "params", "epochs_trained"
]])

## Comparison against linear regression benchmark

The repository already contains `data/lr_benchmark.csv`. The table below compares the selected best hybrid model for each assigned window against that benchmark.

A negative `delta_vs_lr` means that the hybrid model improves the linear regression benchmark.

In [ ]:
lr_path = PROJECT_ROOT / "data" / "lr_benchmark.csv"

if lr_path.exists():
    lr = pd.read_csv(lr_path).rename(columns={
        "MAE_train": "LR_MAE_train",
        "MAE_test": "LR_MAE_test",
    })

    comparison = best_by_window.merge(
        lr[["input_window", "output_window", "LR_MAE_train", "LR_MAE_test"]],
        on=["input_window", "output_window"],
        how="left",
    )

    comparison["delta_vs_lr"] = comparison["MAE_test"] - comparison["LR_MAE_test"]
    comparison["pct_delta_vs_lr"] = 100 * comparison["delta_vs_lr"] / comparison["LR_MAE_test"]

    comparison_path = DATA_OUT / "hybrid_comparison_vs_lr.csv"
    comparison.to_csv(comparison_path, index=False)

    display(comparison[[
        "input_window", "output_window", "model", "MAE_test", "LR_MAE_test", "delta_vs_lr", "pct_delta_vs_lr", "params"
    ]])

    print("Comparison saved to:", comparison_path)
else:
    print("No lr_benchmark.csv found. Benchmark comparison skipped.")

## Test MAE matrix

This matrix is directly usable in the report to summarize the best hybrid model by window.

In [ ]:
matrix = best_by_window.pivot(index="input_window", columns="output_window", values="MAE_test")
matrix_path = DATA_OUT / "hybrid_test_mae_matrix.csv"
matrix.to_csv(matrix_path)

display(matrix)
print("Matrix saved to:", matrix_path)